# Analysis notebook: multilevel runs

Building blocks for plots from every run output under `results/`. The logic lives in `analysis_tools.py` next to this notebook. Every loader returns a list of **runs** with the same layout, so every plot works on every kind of output:

| loader | files | columns (QoIs) |
|---|---|---|
| `load_el_sweep()` | `results/el_sweep/vr_*` (laminate sweep) | tip_deflection, mean_von_mises, compliance |
| `load_vr()` | `results/vr_*` (first laminate VR runs) | same |
| `load_fig9()` | `results/fig9/fig9_rep*` (Fig. 9 replication) | Q; configs `fig9_proposal`, `fig9_state` |
| `load_diagnostics()` | `results/diagnostics/chains_*` (predator-prey sweep) | mean_predator, log_P0 … log_d |
| `load_lr()` | `results/lr_hierarchy/lr_*` (regression hierarchy) | prediction, theta0 … |

A run is a dict: `config`, `rep`, `meta`, `names`, `terms` (`Q_0`, `Y_10`, `Y_21`, …: burned in, aligned), `top` (the fine chain), `reference` (exact value where known).

Core tools: `group(runs, regex)` → `{config: [runs]}`; `plot_se(groups, qoi)`; `term_table(groups, qoi)`; `run_table(runs)`. Each section below is independent: run the setup cell, then any section.

In [ ]:
%matplotlib inline
import os
# os.environ["LIN_EL_RESULTS"] = "/path/to/copied/results"   # only if the results are not in ../results
import numpy as np
import matplotlib.pyplot as plt
import analysis_tools as at

FIGDIR = os.path.join(at.RESULTS, "figures")
os.makedirs(FIGDIR, exist_ok=True)
at.index()

## 1. Laminate sweep: variance reduction per hierarchy

Reproduces `plot_el_vr.py`: one panel per hierarchy, one colour per J, solid multilevel, dashed standard.
`promotion`: `"rand"`, `"fixed"` or `"both"`; `configs`: a regex on the configuration names.

In [ ]:
el = at.load_el_sweep()
at.run_table(el)

In [ ]:
QOI = "compliance"          # tip_deflection, mean_von_mises, compliance
fig = at.plot_el_vr(el, QOI, promotion="rand")
# fig.savefig(os.path.join(FIGDIR, f"el_vr_{QOI}.png"), dpi=200, bbox_inches="tight")

In [ ]:
# all three QoIs, randomised and fixed promotion
for q in el[0]["names"]:
    at.plot_el_vr(el, q, promotion="both")

Per-term variances behind the curves. If Var(Y_ℓ) is close to Var(Q_top), no variance reduction is possible, whatever the curves look like.

In [ ]:
_ = at.term_table(at.group(el, r"_J10_rand"), QOI)

### Custom comparisons

`group(runs, regex)` picks the configurations, and `plot_se` draws them into one axis. Two examples follow: AEM vs no AEM, and two- vs three-level at fixed J.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
at.plot_se(at.group(el, r"^l012_J10_rand"), QOI, ax=axes[0], title="AEM vs no AEM")
at.plot_se(at.group(el, r"_J10_rand$"), QOI, ax=axes[1], title="two vs three levels, J = 10")
fig.tight_layout()

In [ ]:
# final numbers per run: standard and multilevel estimate with SE
for label, runs in at.group(el, r"_J10_rand$").items():
    for r in runs:
        s, m, se_s, se_m = at.estimates(r, QOI)
        print(f"{label} rep {r['rep']}: standard {s:.6e} ± {se_s:.1e}   multilevel {m:.6e} ± {se_m:.1e}")

## 2. First laminate VR runs (`results/vr_*.npz`)

In [ ]:
vr = at.load_vr()
if vr:
    at.run_table(vr)
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.3))
    for ax, q in zip(axes, vr[0]["names"]):
        at.plot_se(at.group(vr), q, ax=ax)
    fig.tight_layout()

## 3. Fig. 9 replication

Proposal- and state-paired corrections from the same runs, plus the RMSE against the reference (open markers) if `results/fig9/reference.npz` exists.

In [ ]:
f9 = at.load_fig9()
if f9:
    _ = at.term_table(at.group(f9), "Q")
    ax = at.plot_se(at.group(f9), "Q", rmse=f9[0]["reference"] is not None, nmin=200)
    ref = os.path.join(at.RESULTS, "fig9", "reference.npz")
    if os.path.exists(ref):
        r = at.load(ref)
        ax.axhline(abs(float(r["Q_state_limit"]) - float(r["Q_ref"])), color="0.4", ls=":", lw=1,
                   label=r"state-paired |bias|, $J\to\infty$")
        ax.legend(fontsize=7)

## 4. Predator-prey sweep (`results/diagnostics`)

`diag_table` reads only the small summaries (all repetitions, fast). The SE-vs-N curves need the chain files, so limit the configs and repetitions: the full sweep is about 8 GB on disk. The parameters are the honest probe here: the model QoI averages over a different time window on each level.

In [ ]:
_ = at.diag_table()              # parameters
_ = at.diag_table(block="qoi")   # mean predator population

In [ ]:
dg = at.load_diagnostics(configs=r"^(base|nsub2|nsub5|fixed_nsub|single)$", reps=5)
at.run_table(dg)

In [ ]:
names = dg[0]["names"] if dg else []
fig, axes = plt.subplots(2, 4, figsize=(20, 8.5))
for ax, q in zip(axes.ravel(), names):
    at.plot_se(at.group(dg), q, ax=ax)
for ax in axes.ravel()[len(names):]:
    ax.remove()
fig.tight_layout()

In [ ]:
# hierarchy choice: does the middle level earn its cost?
dg2 = at.load_diagnostics(configs=r"^(base|two_level_hi|two_level_lo)$", reps=5)
_ = at.term_table(at.group(dg2), "log_a")

## 5. Linear-regression hierarchy

Here the exact posterior mean is known, so `rmse=True` compares the error against the exact value with the SE. `lr_analyse.py` makes the Var(Y_ℓ)-against-n_ℓ figure; the cell below shows the same quantities by hand.

In [ ]:
lr = at.load_lr()
if lr:
    at.plot_se(at.group(lr, r"^deep"), "prediction", rmse=True)

In [ ]:
# Var(Y_l) against n_l, pairs runs (one two-level run per adjacent pair)
pairs = [r for r in lr if r["config"].startswith("pairs")]
if pairs:
    n = np.array([r["sizes"][-1] for r in pairs])
    v = np.array([r["terms"]["Y_10"][:, 0].var() for r in pairs])
    slope = np.polyfit(np.log(n), np.log(v), 1)[0]
    plt.loglog(n, v, "o")
    plt.xlabel("$n_\\ell$"); plt.ylabel("Var($Y_\\ell$)"); plt.title(f"slope {slope:.2f}")

## 6. Laminate level scan (`results/el_level_scan`)

No chains here: QoIs and log-likelihoods of every level at fixed θ. `el_level_scan.py` already saves its figure; this is the raw access.

In [ ]:
import glob
scans = sorted(glob.glob(os.path.join(at.RESULTS, "el_level_scan", "scan_*.npz")))
if scans:
    s = at.load(scans[-1])
    print(os.path.basename(scans[-1]), "  divisions", s["divisions"].tolist())
    h, qoi = s["h"], s["qoi"]                      # qoi: (levels, m, 3)
    for k, name in enumerate(s["meta"]["qoi_names"]):
        var_dq = (qoi[1:, :, k] - qoi[:-1, :, k]).var(axis=1, ddof=1)
        plt.loglog(h[1:], var_dq / qoi[1:, :, k].var(axis=1, ddof=1), "o-", label=name)
    plt.xlabel("$h_\\ell$"); plt.ylabel("Var($Q_\\ell - Q_{\\ell-1}$) / Var($Q_\\ell$)"); plt.legend()

## 7. Raw access

Any file: `at.load(path)`, which returns a dict of arrays with `meta` parsed. Full chains: `at.load_chains(path)` → `{level: {"chain": {"parameters", "qoi", "stats"}, "promoted": {...}, "accepted": flags}}`, with no burn-in applied. The example below shows a trace and the acceptance flags of the finest level.

In [ ]:
chain_files = sorted(glob.glob(os.path.join(at.RESULTS, "**", "chains_*.npz"), recursive=True))
print(len(chain_files), "chain files")
if chain_files:
    c = at.load_chains(chain_files[0])
    top = max(k for k in c if isinstance(k, int))
    theta = c[top]["chain"]["parameters"]
    fig, axes = plt.subplots(theta.shape[1], 1, figsize=(10, 1.6 * theta.shape[1]), sharex=True)
    for j, ax in enumerate(np.atleast_1d(axes)):
        ax.plot(theta[:, j], lw=0.5)
    np.atleast_1d(axes)[0].set_title(f"{os.path.basename(chain_files[0])}, level {top}: "
                                     f"acceptance {c[top]['accepted'].mean():.3f}, complete: {bool(c['complete'])}")